<a href="https://colab.research.google.com/github/LoredanaIonescu/product-category-prediction/blob/main/exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
# ================================
# 1. IMPORT LIBRARII
# ================================

import pandas as pd
import numpy as np
import re

# librării pentru ML
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB

from sklearn.metrics import classification_report, accuracy_score

from google.colab import files
uploaded = files.upload()
# ================================
# 2. LOAD DATA
# ================================

# citim datasetul
df = pd.read_csv("products.csv")

# vedem primele rânduri
df.head()

Saving products.csv to products (1).csv


,product ID,Product Title,Merchant ID,Category Label,_Product Code,Number_of_Views,Merchant Rating,Listing Date
0,1,apple iphone 8 plus 64gb silver,1,Mobile Phones,QA-2276-XC,860.0,2.5,05/10/2024
1,2,apple iphone 8 plus 64 gb spacegrau,2,Mobile Phones,KA-2501-QO,3772.0,4.8,12/31/2024
2,3,apple mq8n2b/a iphone 8 plus 64gb 5.5 12mp sim...,3,Mobile Phones,FP-8086-IE,3092.0,3.9,11/10/2024
3,4,apple iphone 8 plus 64gb space grey,4,Mobile Phones,YI-0086-US,466.0,3.4,05/02/2022
4,5,apple iphone 8 plus gold 5.5 64gb 4g unlocked ...,5,Mobile Phones,NZ-3586-WP,4426.0,1.6,04/12/2023


In [20]:
# ================================
# 3. EXPLORARE DATE
# ================================

# dimensiunea datasetului
print("Shape:", df.shape)

# coloanele existente
print("Columns:", df.columns)

# distribuția categoriilor (important pentru dezechilibru)
print(df["Category Label"].value_counts())

Shape: (35311, 8)
Columns: Index(['product ID', 'Product Title', 'Merchant ID', 'Category Label',
       '_Product Code', 'Number_of_Views', 'Merchant Rating',
       ' Listing Date  '],
      dtype='object')
Category Label
Fridge Freezers     5495
Washing Machines    4036
Mobile Phones       4020
CPUs                3771
TVs                 3564
Fridges             3457
Dishwashers         3418
Digital Cameras     2696
Microwaves          2338
Freezers            2210
fridge               123
CPU                   84
Mobile Phone          55
Name: count, dtype: int64


In [21]:
#Normalizarea categoriilor


def normalize_category(cat):
    cat = str(cat).lower().strip()

    mapping = {
        "fridge": "Fridges",
        "fridges": "Fridges",
        "fridge freezers": "Fridge Freezers",

        "cpu": "CPUs",
        "cpus": "CPUs",

        "mobile phone": "Mobile Phones",
        "mobile phones": "Mobile Phones"
    }

    return mapping.get(cat, cat.title())



df["Category Label"] = df["Category Label"].apply(normalize_category)
print(df["Category Label"].value_counts())

Category Label
Fridge Freezers     5495
Mobile Phones       4075
Washing Machines    4036
CPUs                3855
Fridges             3580
Tvs                 3564
Dishwashers         3418
Digital Cameras     2696
Microwaves          2338
Freezers            2210
Nan                   44
Name: count, dtype: int64


In [22]:
# ================================
# 4. CURĂȚARE TEXT
# ================================

def clean_text(text):
    """
    Curăță textul:
    - lowercase
    - elimină caractere speciale
    - elimină spații multiple
    """
    text = str(text).lower()  # transformăm în lowercase
    text = re.sub(r'[^a-z0-9\s]', '', text)  # scoatem simboluri
    text = re.sub(r'\s+', ' ', text).strip()  # spații curate
    return text

# aplicăm curățarea pe titlu
df["clean_title"] = df["Product Title"].apply(clean_text)

In [23]:
# ================================
# 5. TRATARE VALORI LIPSĂ
# ================================

# eliminăm rândurile fără titlu sau categorie
df = df.dropna(subset=["clean_title", "Category Label"])


In [24]:
# ================================
# 6. FEATURE ENGINEERING
# ================================

# lungimea titlului
df["title_length"] = df["clean_title"].apply(len)

# număr de cuvinte
df["word_count"] = df["clean_title"].apply(lambda x: len(x.split()))

In [25]:
# ================================
# 7. SPLIT DATE
# ================================

# X = input (titlul produsului)
# y = target (categoria)
X = df["clean_title"]
y = df["Category Label"]

# împărțim în train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [26]:
# ================================
# 8. MODEL 1: LOGISTIC REGRESSION
# ================================

# pipeline = vectorizare + model
model_lr = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000)),  # transformă text în vectori
    ("clf", LogisticRegression(max_iter=1000))      # model de clasificare
])

# antrenare
model_lr.fit(X_train, y_train)

# predicții
pred_lr = model_lr.predict(X_test)

# evaluare
print("=== Logistic Regression ===")
print("Accuracy:", accuracy_score(y_test, pred_lr))
print(classification_report(y_test, pred_lr))



=== Logistic Regression ===
Accuracy: 0.9469064137052244
                  precision    recall  f1-score   support

            CPUs       1.00      0.99      0.99       771
 Digital Cameras       1.00      0.99      1.00       542
     Dishwashers       0.88      0.94      0.91       661
        Freezers       0.97      0.91      0.94       437
 Fridge Freezers       0.91      0.92      0.92      1113
         Fridges       0.89      0.89      0.89       727
      Microwaves       0.98      0.94      0.96       469
   Mobile Phones       0.96      0.99      0.98       816
             Nan       0.00      0.00      0.00        11
             Tvs       0.97      0.98      0.98       723
Washing Machines       0.94      0.94      0.94       793

        accuracy                           0.95      7063
       macro avg       0.86      0.86      0.86      7063
    weighted avg       0.95      0.95      0.95      7063



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [28]:
# ================================
# 9. MODEL 2: NAIVE BAYES
# ================================

model_nb = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000)),
    ("clf", MultinomialNB())
])

model_nb.fit(X_train, y_train)

pred_nb = model_nb.predict(X_test)

print("=== Naive Bayes ===")
print("Accuracy:", accuracy_score(y_test, pred_nb))
print(classification_report(y_test, pred_nb))

=== Naive Bayes ===
Accuracy: 0.9195809146255133
                  precision    recall  f1-score   support

            CPUs       1.00      0.99      0.99       771
 Digital Cameras       0.99      1.00      0.99       542
     Dishwashers       0.86      0.94      0.90       661
        Freezers       0.98      0.58      0.73       437
 Fridge Freezers       0.78      0.93      0.85      1113
         Fridges       0.88      0.80      0.84       727
      Microwaves       0.99      0.94      0.96       469
   Mobile Phones       0.99      0.99      0.99       816
             Nan       0.00      0.00      0.00        11
             Tvs       0.97      0.97      0.97       723
Washing Machines       0.94      0.95      0.95       793

        accuracy                           0.92      7063
       macro avg       0.85      0.83      0.83      7063
    weighted avg       0.92      0.92      0.92      7063



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [29]:
# ================================
# 10. SALVARE MODEL FINAL
# ================================

import pickle

# alegem modelul mai bun (Logistic Regression)
with open("model.pkl", "wb") as f:
    pickle.dump(model_lr, f)

print("Model salvat cu succes!")

Model salvat cu succes!
